# 6 Scaling: From One Document to a Corpus

In Sections 3 and 4, we built a complete pipeline: ingestion with Docling, word-boundary-respecting chunking, embedding with Granite, retrieval from ChromaDB, and grounded generation. In Section 5, we evaluated the results and proved that retrieval adds real, measurable value.

All of that was built against a single document. One rulebook. One PDF.

No customer has one document.

They have twelve. Or two hundred. Or twelve thousand. They have policy manuals that reference procedure guides that reference training materials that contradict last year's version of themselves. They have documents written by different people in different decades using different terminology for the same concepts.

The question we need to answer now is not whether the pipeline works. We proved that. The question is whether it survives.


## 6.1 The System, Not the Step
Up to this point, we have worked through a progression that felt sequential: ingest a document, chunk it, embed it, retrieve from it, evaluate the results. Each step had a purpose. Each step taught something.

But here is the shift that matters: those were not steps. They were pipeline stages.

The distinction is important. A step is something you do once. A pipeline is something that runs again, against new documents, against updated documents, against documents that did not exist when you first built the system. The customer's corpus is not static. Their PDFs change. New policies get published. Old manuals get revised. The model might get swapped out. The embedding strategy might need tuning.

If the system cannot survive any of those changes, it is not a system. It is a demo that worked once.

The answer, and this is the key insight for the field: the pipeline does not change. The input does.

The same Docling conversion, the same word-boundary-respecting chunking, the same Granite embedding model, the same ChromaDB vector store, the same retrieval function. All of it carries forward unchanged. What changes is the loop: instead of processing one PDF, we iterate over a directory of them.

That is not a trivial observation. It means the architectural decisions we made in Sections 3 and 4 (the chunking strategy, the overlap size, the embedding model) are now load-bearing. They apply to every document in the corpus. If they were wrong for one document, they will be wrong for all of them.

>Facilitator note: This is where you land the reframe. Ingestion was not a task. Chunking was not a tweak. Evaluation was not an afterthought. They were all pipeline stages. And now we see them working together.

## 6.2 Making the Workflow Explicit

Now we name the system. Walk through the end-to-end flow in plain language before running any code:

1. Source documents enter the system (PDFs in a directory)
1. Each document is parsed and normalized (Docling converts to Markdown)
1. Content is structured and chunked (word-boundary-respecting splits with overlap)
1. Chunks are embedded (Granite embedding model, loaded once, reused for all documents)
1. Everything is loaded into a vector store (ChromaDB, with metadata tracking provenance)
1. The model is exercised against the corpus (RAG queries with source attribution)
1. Outputs are evaluated (same question set from Section 5, re-run against the new collection)
1. Decisions are made about what, if anything, needs to change

Nothing here is exotic. Every stage is observable. Every stage is replaceable. If the chunking strategy is not working, you swap it. If the embedding model is not capturing domain nuance, you try another. If Docling mishandles a particular document layout, you investigate and fix the ingestion. The pipeline gives you a place to look when something goes wrong.

>Field takeaway: If you cannot point to where something went wrong, you do not have a pipeline. You have a demo.

The code that follows is deliberately familiar. The helper functions are identical to Section 4. The Docling configuration is the same. The embedding model is the same. What is new is the orchestration: a loop that processes every PDF in a directory and accumulates the results into a single, searchable collection. Each chunk carries metadata that records which document it came from. That single design decision is what makes source attribution possible downstream.

### 6.2.1 Setup MaaS Connection
Setup code to connect to MaaS endpoint.

In [4]:
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base

import requests
import json
import numpy as np
import chromadb
from pathlib import Path
from sentence_transformers import SentenceTransformer
from docling.document_converter import DocumentConverter
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat
from docling.document_converter import PdfFormatOption

url_chat = f"{endpoint_base}/chat/completions"
model_id = "granite-3-2-8b-instruct"

print("\u2705 All imports loaded")

ImportError: cannot import name 'PreTrainedModel' from 'transformers' (/home/frank/jupyterlab/.venv/lib/python3.12/site-packages/transformers/__init__.py)

### 6.2.2 Helper functions
These are the same `chunk_text_markdown` and `add_to_collection` functions from Section 4. Unchanged. The pipeline is the same; we are just feeding it more documents.



In [ ]:
def chunk_text_markdown(text, chunk_size=1000, overlap=200):
    import re
    
    # Split on markdown headings (keep the heading with the text that follows)
    # and on double newlines (paragraph boundaries)
    sections = re.split(r'(?=\n#{1,6} )', text)
    
    # Further split each section on paragraph breaks (double newline)
    blocks = []
    for section in sections:
        paragraphs = re.split(r'\n{2,}', section)
        for p in paragraphs:
            stripped = p.strip()
            if stripped:
                blocks.append(stripped)
    
    # Accumulate blocks into chunks
    chunks = []
    current_chunk = ""
    
    for block in blocks:
        # If a single block exceeds chunk_size, split it at word boundaries
        if len(block) > chunk_size:
            # Flush current chunk first
            if current_chunk.strip():
                chunks.append(current_chunk.strip())
                current_chunk = ""
            
            # Word-boundary fallback for oversized blocks
            start = 0
            while start < len(block):
                end = start + chunk_size
                if end < len(block):
                    space_pos = block.rfind(" ", start, end)
                    if space_pos > start:
                        end = space_pos
                sub = block[start:end].strip()
                if sub:
                    chunks.append(sub)
                next_start = end - overlap
                start = max(next_start, start + 1)
            continue
        
        # Would adding this block exceed the budget?
        candidate = (current_chunk + "\n\n" + block).strip() if current_chunk else block
        if len(candidate) > chunk_size and current_chunk.strip():
            chunks.append(current_chunk.strip())
            current_chunk = block
        else:
            current_chunk = candidate
    
    # Flush the last chunk
    if current_chunk.strip():
        chunks.append(current_chunk.strip())
    
    # Apply overlap: prepend tail of previous chunk to each subsequent chunk
    if overlap > 0 and len(chunks) > 1:
        overlapped = [chunks[0]]
        for i in range(1, len(chunks)):
            prev = chunks[i - 1]
            # Take the last `overlap` characters, back up to word boundary
            if len(prev) > overlap:
                tail_start = len(prev) - overlap
                space_pos = prev.find(" ", tail_start)
                if space_pos != -1:
                    tail = prev[space_pos:].strip()
                else:
                    tail = prev[tail_start:].strip()
            else:
                tail = prev.strip()
            overlapped.append((tail + "\n\n" + chunks[i]).strip())
        chunks = overlapped
    
    return chunks


In [ ]:
def add_to_collection(collection, chunks, embeddings, metadatas=None, ids=None, batch_size=5000):
    """Batch-load chunks into a ChromaDB collection."""
    for i in range(0, len(chunks), batch_size):
        end = min(i + batch_size, len(chunks))
        batch_kwargs = {
            "documents": chunks[i:end],
            "embeddings": embeddings[i:end].tolist() if hasattr(embeddings, 'tolist') else embeddings[i:end],
            "ids": ids[i:end] if ids else [f"chunk_{j}" for j in range(i, end)],
        }
        if metadatas:
            batch_kwargs["metadatas"] = metadatas[i:end]
        collection.add(**batch_kwargs)
    print(f"\u2705 Loaded {collection.count()} chunks into the vector store")

### 6.2.3 Discover the Corpus
The pipeline scans the `docs/` directory for every PDF. If you have placed additional documents there, they will be picked up automatically. The pipeline is input-agnostic.


In [ ]:
docs_dir = Path("../docs")
pdf_files = sorted([
    f for f in docs_dir.rglob("*.pdf")
    if ".ipynb_checkpoints" not in str(f)
])

print(f"Found {len(pdf_files)} PDFs:")
for f in pdf_files:
    print(f"  {f.relative_to(docs_dir)}")


> Note: I added the checkpoint filter. That .ipynb_checkpoints PDF in your current output would cause garbage during conversion.

### 6.2.4 Load the Embedding Model
We load the embedding model once and reuse it for all documents. Same Granite Embedding model from Section 4.


In [2]:
print("Loading Granite embedding model...")
embed_model = SentenceTransformer("../models/granite-embedding-30m-english")
print("\u2705 Embedding model loaded")

Loading Granite embedding model...


NameError: name 'SentenceTransformer' is not defined

### 6.2.5 Configure Docling Converter
Same configuration as Section 3. OCR is disabled because all documents in this lab contain extractable text. In a production deployment with scanned documents, you would enable OCR here and the rest of the pipeline would not change.


In [3]:
pdf_options = PdfPipelineOptions()
pdf_options.do_ocr = False

converter = DocumentConverter(
    format_options={
        InputFormat.PDF: PdfFormatOption(pipeline_options=pdf_options)
    }
)
print("\u2705 Docling converter configured (OCR disabled)")

NameError: name 'PdfPipelineOptions' is not defined

## 6.3 Run the Pipeline
This is the same sequence from Sections 3 and 4, inside a loop. For each PDF: convert, chunk, embed, and accumulate. Every chunk carries metadata with its source filename, and every chunk ID encodes its origin document. That is not decorative. It is what makes source attribution possible at query time.


>Facilitator note: This cell will take a few minutes depending on how many PDFs are in the directory. Watch the progress output. If it stalls on a particular document for more than three minutes, note which file caused the delay and move to pre-built outputs. The learning objective does not require every PDF to finish processing.


In [7]:
from chromadb.errors import NotFoundError

client = chromadb.PersistentClient(path="../chroma_db")

# Delete if exists (safe for re-runs)
try:
    client.delete_collection("rpg_rules_multi")
except (ValueError, NotFoundError):
    pass

collection_multi = client.create_collection(
    name="rpg_rules_multi",
    metadata={"hnsw:space": "cosine"}
)

for pdf_idx, pdf_path in enumerate(pdf_files, 1):
    filename = pdf_path.name
    print(f"\n{'='*60}")
    print(f"[{pdf_idx}/{len(pdf_files)}] Processing: {filename}")
    print(f"{'='*60}")

    # Step 1: Convert PDF to Markdown
    print("  Converting PDF to markdown...")
    result = converter.convert(str(pdf_path))
    markdown_text = result.document.export_to_markdown()
    print(f"  Extracted {len(markdown_text):,} characters")

    # Step 2: Chunk
    chunks = chunk_text_markdown(markdown_text, chunk_size=1000, overlap=200)
    print(f"  Created {len(chunks)} chunks")

    # Step 3: Embed
    print("  Embedding chunks...")
    embeddings = embed_model.encode(chunks, show_progress_bar=False, batch_size=64)

    # Step 4: Load directly into ChromaDB
    metadatas = [{"source": filename} for _ in chunks]
    ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]

    add_to_collection(
        collection_multi,
        chunks,
        embeddings,
        metadatas=metadatas,
        ids=ids
    )

    print(f"  \u2705 Done \u2014 collection now has {collection_multi.count()} chunks")

print(f"\n{'='*60}")
print(f"Pipeline complete: {collection_multi.count()} total chunks from {len(pdf_files)} document(s)")
print(f"{'='*60}")

2026-02-18 10:26:32,799 - INFO - Anonymized telemetry enabled. See                     https://docs.trychroma.com/telemetry for more information.
2026-02-18 10:26:32,887 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-02-18 10:26:32,978 - INFO - Going to convert document batch...
2026-02-18 10:26:32,979 - INFO - Initializing pipeline for StandardPdfPipeline with options hash 75463f421d05cb4304e1f714cf00d35d
2026-02-18 10:26:32,984 - INFO - Loading plugin 'docling_defaults'
2026-02-18 10:26:32,985 - INFO - Registered picture descriptions: ['vlm', 'api']
2026-02-18 10:26:32,991 - INFO - Loading plugin 'docling_defaults'
2026-02-18 10:26:32,992 - INFO - Registered ocr engines: ['auto', 'easyocr', 'ocrmac', 'rapidocr', 'tesserocr', 'tesseract']
2026-02-18 10:26:32,994 - INFO - Accelerator device: 'cpu'



[1/6] Processing: Basic-Fantasy-RPG-Rules-r107-bookmarked.pdf
  Converting PDF to markdown...


2026-02-18 10:26:33,546 - INFO - Accelerator device: 'cpu'
2026-02-18 10:26:33,761 - INFO - Processing document Basic-Fantasy-RPG-Rules-r107-bookmarked.pdf
2026-02-18 10:39:30,155 - INFO - Finished converting document Basic-Fantasy-RPG-Rules-r107-bookmarked.pdf in 777.29 sec.


  Extracted 757,271 characters
  Created 995 chunks
  Embedding chunks...
✅ Loaded 995 chunks into the vector store
  ✅ Done — collection now has 995 chunks

[2/6] Processing: Basic-Fantasy-RPG-Rules-r107-lite.pdf
  Converting PDF to markdown...


2026-02-18 10:40:21,874 - INFO - detected formats: [<InputFormat.PDF: 'pdf'>]
2026-02-18 10:40:21,920 - INFO - Going to convert document batch...
2026-02-18 10:40:21,922 - INFO - Processing document Basic-Fantasy-RPG-Rules-r107-lite.pdf


KeyboardInterrupt: 

## 6.1 Multi-Document Pipeline

The key architectural decisions for multi-document ingestion:

1. **Unique chunk IDs** — each chunk needs an ID that encodes its source document, so we can trace any retrieval result back to its origin.
2. **Metadata tracking** — every chunk carries `{"source": filename}` so retrieval results include provenance.
3. **New collection** — we use `rpg_rules_multi` to keep this separate from the single-document collection in Section 4.

> **Facilitator note:** If you've placed additional PDFs in `docs/`, they'll be picked up automatically. The pipeline is input-agnostic.

In [11]:
from pathlib import Path

docs_dir = Path("../docs")
pdf_files = sorted(docs_dir.rglob("*.pdf"))

print(f"Found {len(pdf_files)} PDFs:")
for f in pdf_files:
    print(f"  {f.relative_to(docs_dir)}")

Found 7 PDFs:
  .ipynb_checkpoints/Basic-Fantasy-RPG-Rules-r142-checkpoint.pdf
  3rdEdition/Basic-Fantasy-RPG-Rules-r107-bookmarked.pdf
  3rdEdition/Basic-Fantasy-RPG-Rules-r107-lite.pdf
  3rdEdition/Basic-Fantasy-RPG-Rules-r107.pdf
  BFRPG-Monster-Index-r7.pdf
  Basic-Fantasy-RPG-Rules-r142.pdf
  EE1-Equipment-Emporium-r33.pdf


### Load the Embedding Model

We load the embedding model once and reuse it for all documents. Same Granite Embedding model from Section 4.

In [ ]:
print("Loading Granite embedding model...")
embed_model = SentenceTransformer("../models/granite-embedding-30m-english")
print("\u2705 Embedding model loaded")

In [ ]:
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.datamodel.base_models import InputFormat

pipeline_options = PdfPipelineOptions(do_ocr=False)
converter = DocumentConverter(
    format_options={InputFormat.PDF: {"pipeline_options": pipeline_options}}
)
print("\u2705 Docling converter configured (OCR disabled)")

In [ ]:
all_chunks = []
all_embeddings = []
all_metadatas = []
all_ids = []

for pdf_idx, pdf_path in enumerate(pdf_files, 1):
    filename = pdf_path.name
    print(f"\n{'='*60}")
    print(f"[{pdf_idx}/{len(pdf_files)}] Processing: {filename}")
    print(f"{'='*60}")

    # Step 1: Convert PDF to markdown with Docling
    print("  Converting PDF to markdown...")
    result = converter.convert(str(pdf_path))
    markdown_text = result.document.export_to_markdown()
    print(f"  Extracted {len(markdown_text):,} characters")

    # Step 2: Chunk the text
    chunks = chunk_text(markdown_text, chunk_size=1000, overlap=200)
    print(f"  Created {len(chunks)} chunks")

    # Step 3: Embed all chunks
    print("  Embedding chunks...")
    embeddings = embed_model.encode(chunks, show_progress_bar=False, batch_size=64)

    # Step 4: Build metadata and IDs with source attribution
    metadatas = [{"source": filename} for _ in chunks]
    ids = [f"{filename}_chunk_{i}" for i in range(len(chunks))]

    all_chunks.extend(chunks)
    all_embeddings.append(embeddings)
    all_metadatas.extend(metadatas)
    all_ids.extend(ids)

    print(f"  \u2705 Done \u2014 running total: {len(all_chunks)} chunks from {pdf_idx} document(s)")

# Stack all embeddings into a single array
all_embeddings = np.vstack(all_embeddings)

print(f"\n{'='*60}")
print(f"Pipeline complete: {len(all_chunks)} total chunks from {len(pdf_files)} document(s)")
print(f"Embedding matrix shape: {all_embeddings.shape}")
print(f"{'='*60}")

### 6.1.5 Load into ChromaDB

We create a new collection called `rpg_rules_multi`. This keeps it separate from the single-document `rpg_rules` collection built in Section 4, so we can compare them later.

Every chunk now carries metadata with its source filename — this is what enables source attribution at query time.

In [ ]:
client = chromadb.PersistentClient(path="../chroma_db")

# Delete if exists (safe for re-runs)
try:
    client.delete_collection("rpg_rules_multi")
except ValueError:
    pass

collection_multi = client.create_collection(
    name="rpg_rules_multi",
    metadata={"hnsw:space": "cosine"}
)

add_to_collection(
    collection_multi,
    all_chunks,
    all_embeddings,
    metadatas=all_metadatas,
    ids=all_ids
)

print(f"Collection '{collection_multi.name}' — {collection_multi.count()} chunks")

## 6.2 Retrieval with Source Attribution

In a single-document pipeline, you know where every answer comes from — there's only one source. With multiple documents, the customer will always ask:

> *"Where did that come from?"*

Source attribution isn't a nice-to-have. It's the difference between a demo and a deployable system. The function below extends our RAG query to return the source document for each retrieved chunk alongside the answer.

In [ ]:
def ask_with_rag_sources(question, collection, embed_model, n_results=3):
    """RAG query that returns the answer plus source attribution for each chunk."""
    # Retrieve with metadata
    query_embedding = embed_model.encode(question).tolist()
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results,
        include=["documents", "metadatas"]
    )

    chunks = results["documents"][0]
    metadatas = results["metadatas"][0]
    sources = [(chunk, meta.get("source", "unknown")) for chunk, meta in zip(chunks, metadatas)]

    # Build grounded prompt
    context = "\n\n---\n\n".join(chunks)
    system_msg = (
        "Answer the question using only the provided context. "
        "If the context does not contain enough information, say so."
    )
    headers = {
        "Authorization": f"Bearer {key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model_id,
        "messages": [
            {"role": "system", "content": system_msg},
            {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {question}"}
        ],
        "max_tokens": 300,
        "temperature": 0
    }

    response = requests.post(url_chat, headers=headers, json=data)
    response.raise_for_status()
    answer = response.json()["choices"][0]["message"]["content"]

    return answer, sources

In [ ]:
test_question = "What happens if a Thief fails an Open Locks attempt?"
answer, sources = ask_with_rag_sources(test_question, collection_multi, embed_model)

print(f"Question: {test_question}\n")
print(f"Answer:\n{answer}\n")
print("=" * 60)
print("SOURCE ATTRIBUTION")
print("=" * 60)
for i, (chunk_text_preview, source) in enumerate(sources, 1):
    print(f"\n--- Chunk {i} | Source: {source} ---")
    print(chunk_text_preview[:200] + "...")

## 6.3 Evaluation: Same Questions, Multi-Document Corpus

We re-run the same 5 evaluation questions from Section 5. This tells us whether the multi-document pipeline produces the same quality answers — and whether source attribution is consistent.

In [ ]:
eval_questions = [
    {
        "id": "q1",
        "question": "What happens if a Thief fails an Open Locks attempt?",
        "reference_answer": "The Thief must wait until they have gained another level of experience before trying again. It may only be tried once per lock."
    },
    {
        "id": "q2",
        "question": "Why can't Elves roll higher than a d6 for hit points?",
        "reference_answer": "Elves use a d6 for hit dice because they are a combination Fighter/Magic-User class, and their hit die reflects the Magic-User side."
    },
    {
        "id": "q3",
        "question": "Can a Fighter use magic-user scrolls?",
        "reference_answer": "No. Only spell casters can use magic-user scrolls."
    },
    {
        "id": "q4",
        "question": "How is initiative determined in combat?",
        "reference_answer": "Each side rolls 1d6 at the beginning of each round. The side with the highest roll acts first."
    },
    {
        "id": "q5",
        "question": "What is the maximum number of retainers a character can hire?",
        "reference_answer": "The number of retainers is based on the character's Charisma bonus. A character with average Charisma can hire up to 4."
    },
]

print(f"Evaluation set: {len(eval_questions)} questions")
for q in eval_questions:
    print(f"  [{q['id']}] {q['question']}")

In [ ]:
eval_results = []

print("Running evaluation against multi-document collection...\n")
for q in eval_questions:
    print(f"[{q['id']}] {q['question']}")
    answer, sources = ask_with_rag_sources(q["question"], collection_multi, embed_model)
    eval_results.append({
        "id": q["id"],
        "question": q["question"],
        "reference_answer": q["reference_answer"],
        "answer": answer,
        "sources": sources
    })
    print(f"  \u2705 Done\n")

print(f"All {len(eval_results)} questions complete.")

In [ ]:
print("=" * 80)
print("MULTI-DOCUMENT EVALUATION RESULTS")
print("=" * 80)

for r in eval_results:
    print(f"\n{'\u2500' * 80}")
    print(f"{r['id'].upper()}: {r['question']}")
    print(f"{'\u2500' * 80}")

    print(f"\n\U0001f4d6 Reference:\n   {r['reference_answer']}")
    print(f"\n\U0001f4da RAG Answer (multi-doc):\n   {r['answer']}")

    print(f"\n   Sources:")
    source_files = set(src for _, src in r["sources"])
    for src in source_files:
        print(f"     - {src}")

    print(f"\n   Retrieved chunks:")
    for i, (chunk_preview, source) in enumerate(r["sources"], 1):
        print(f"     [{i}] ({source}) {chunk_preview[:120]}...")

## 6.4 Comparison: Single-Document vs Multi-Document

Now we compare the `rpg_rules` collection from Section 4 (single document) against `rpg_rules_multi` (full corpus). This reveals what changes when you scale:

- **Redundancy** — multiple documents may contain overlapping information, leading to duplicate or near-duplicate chunks in retrieval
- **Noise** — more documents means more chunks competing for the top-N retrieval slots; irrelevant chunks from unrelated documents can displace relevant ones
- **Conflict** — different documents may state different rules, versions, or interpretations of the same concept

In [ ]:
# Load the single-doc collection from Section 4
try:
    collection_single = client.get_collection(name="rpg_rules")
    print(f"Single-doc collection: {collection_single.count()} chunks")
    print(f"Multi-doc collection:  {collection_multi.count()} chunks")
except Exception as e:
    print(f"\u26a0\ufe0f  Could not load single-doc collection 'rpg_rules': {e}")
    print("   Run Section 4 first to create it.")
    collection_single = None

if collection_single:
    compare_question = "How is initiative determined in combat?"

    print(f"\n{'=' * 70}")
    print(f"COMPARISON: {compare_question}")
    print(f"{'=' * 70}")

    # Single-doc answer (no source metadata)
    q_emb = embed_model.encode(compare_question).tolist()
    single_results = collection_single.query(
        query_embeddings=[q_emb], n_results=3
    )
    single_context = "\n\n---\n\n".join(single_results["documents"][0])
    headers = {"Authorization": f"Bearer {key}", "Content-Type": "application/json"}
    single_data = {
        "model": model_id,
        "messages": [
            {"role": "system", "content": "Answer the question using only the provided context. If the context does not contain enough information, say so."},
            {"role": "user", "content": f"Context:\n{single_context}\n\nQuestion: {compare_question}"}
        ],
        "max_tokens": 300, "temperature": 0
    }
    resp = requests.post(url_chat, headers=headers, json=single_data)
    single_answer = resp.json()["choices"][0]["message"]["content"]

    # Multi-doc answer (with source attribution)
    multi_answer, multi_sources = ask_with_rag_sources(
        compare_question, collection_multi, embed_model
    )

    print(f"\n\U0001f4c4 Single-document RAG ({collection_single.count()} chunks):")
    print(f"   {single_answer}")

    print(f"\n\U0001f4da Multi-document RAG ({collection_multi.count()} chunks):")
    print(f"   {multi_answer}")

    print(f"\n   Sources used:")
    for _, source in multi_sources:
        print(f"     - {source}")

## 6.5 Takeaways

1. **The pipeline is input-agnostic.** The same chunking, embedding, and retrieval code processes one document or a hundred. What changes is the input list, not the architecture.

2. **Source attribution isn't optional.** The moment you move beyond a single document, every answer needs to say where it came from. Without this, the system is a black box — and no customer will deploy a black box.

3. **Scaling reveals data management problems, not model problems.** Conflicting information across documents, version drift, duplicate content — these are corpus curation issues. The model can only work with what retrieval gives it.

4. **The evaluation framework is now essential.** With a single document, you could eyeball quality. With a corpus, you need structured evaluation (Section 5) to catch regressions. If adding a new document makes existing answers worse, you need to know immediately.

> *"The pipeline scales. The question is whether your data does."*